In [1]:
import pandas as pd
import numpy as np

from scipy import stats

In [4]:
orders = pd.read_csv(
    "../Data/cleaned/orders_featured.csv"
)

order_items = pd.read_csv(
    "../Data/cleaned/order_items_featured.csv"
)

reviews = pd.read_csv(
    "../Data/cleaned/reviews_featured.csv"
)

payments = pd.read_csv(
    "../Data/cleaned/payments_clean.csv"
)

products = pd.read_csv(
    "../Data/cleaned/products_clean.csv"
)

print("Datasets loaded successfully!")

print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Reviews:", reviews.shape)
print("Payments:", payments.shape)
print("Products:", products.shape)
print(orders.columns.tolist())

Datasets loaded successfully!
Orders: (99441, 23)
Order Items: (112650, 8)
Reviews: (99224, 8)
Payments: (103886, 5)
Products: (32951, 9)
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'purchase_year', 'purchase_month', 'purchase_month_name', 'purchase_day_of_week', 'delivery_days', 'estimated_delivery_days', 'delay_days', 'is_late', 'purchase_hour', 'total_order_value_x', 'total_order_value_y', 'total_order_value', 'total_items_x', 'total_items_y', 'total_items']


TEST 1 — T-TEST

In [6]:
t_test_data = orders[
    [
        "order_id",
        "is_late",
        "delivery_days",
        "order_delivered_customer_date"
    ]
].merge(
    reviews[
        [
            "order_id",
            "review_score"
        ]
    ],
    on="order_id",
    how="inner"
)

t_test_data.head()

,order_id,is_late,delivery_days,order_delivered_customer_date,review_score
0,e481f51cbdc54678b7cc49136f2d6af7,False,8.44,2017-10-10 21:25:13,4
1,53cdb2fc8bc7dce0b6741e2150273451,False,13.78,2018-08-07 15:27:45,4
2,47770eb9100c2d0c44946d9cf07ec65d,False,9.39,2018-08-17 18:06:29,5
3,949d5b44dbf5de918fe9c16f97b45f8a,False,13.21,2017-12-02 00:28:42,5
4,ad21c59c0840e6cb83a9ceb5573f8159,False,2.87,2018-02-16 18:17:02,5


In [14]:
t_test_data = t_test_data[
    t_test_data["is_late"].notna()
    & t_test_data["review_score"].notna()
]
print(t_test_data.shape)

on_time_reviews = t_test_data[
    t_test_data["is_late"] == False
]["review_score"]

late_reviews = t_test_data[
    t_test_data["is_late"] == True
]["review_score"]

print("On-time reviews:", len(on_time_reviews))
print("Late reviews:", len(late_reviews))

print(
    "Average review score - On Time:",
    round(on_time_reviews.mean(), 2)
)

print(
    "Average review score - Late:",
    round(late_reviews.mean(), 2)
)

t_statistic, p_value = stats.ttest_ind(
    on_time_reviews,
    late_reviews,
    equal_var=False
)

print("T-statistic:", t_statistic)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print(
        "Result: The difference in average review scores "
        "is statistically significant at the 5% level."
    )
else:
    print(
        "Result: The difference in average review scores "
        "is not statistically significant at the 5% level."
    )

(96359, 5)
On-time reviews: 88658
Late reviews: 7701
Average review score - On Time: 4.29
Average review score - Late: 2.57
T-statistic: 89.55081413351807
P-value: 0.0
Result: The difference in average review scores is statistically significant at the 5% level.


TEST 2 — ANOVA

In [24]:
anova_data = order_items.merge(
    products[
        [
            "product_id",
            "product_category_name"
        ]
    ],
    on="product_id",
    how="left"
)
anova_data[
    [
        "product_id",
        "product_category_name",
        "price"
    ]
].head()
anova_data[
    [
        "product_id",
        "product_category_name",
        "price"
    ]
].head()

category_counts = (
    anova_data["product_category_name"]
    .value_counts()
)

print(category_counts.head(20))

top_categories = (
    category_counts
    .head(10)
    .index
)

anova_data_top = anova_data[
    anova_data["product_category_name"]
    .isin(top_categories)
]

category_means = (
    anova_data_top
    .groupby("product_category_name")["price"]
    .mean()
    .sort_values(ascending=False)
)

category_means

anova_groups = [
    group["price"].dropna().values
    for _, group in anova_data_top.groupby(
        "product_category_name"
    )
]
print("Number of groups:", len(anova_groups))
f_statistic, p_value = stats.f_oneway(
    *anova_groups
)

print("F-statistic:", f_statistic)
print("P-value:", p_value)

alpha = 0.05

if p_value < alpha:
    print(
        "Result: There is a statistically significant "
        "difference in mean prices among the selected "
        "product categories."
    )
else:
    print(
        "Result: There is no statistically significant "
        "difference in mean prices among the selected "
        "product categories."
    )

product_category_name
cama_mesa_banho                11115
beleza_saude                    9670
esporte_lazer                   8641
moveis_decoracao                8334
informatica_acessorios          7827
utilidades_domesticas           6964
relogios_presentes              5991
telefonia                       4545
ferramentas_jardim              4347
automotivo                      4235
brinquedos                      4117
cool_stuff                      3796
perfumaria                      3419
bebes                           3065
eletronicos                     2767
papelaria                       2517
fashion_bolsas_e_acessorios     2031
pet_shop                        1947
moveis_escritorio               1691
unknown                         1603
Name: count, dtype: int64
Number of groups: 10
F-statistic: 310.29875832410073
P-value: 0.0
Result: There is a statistically significant difference in mean prices among the selected product categories.


TEST 3 — CHI-SQUARE TEST

In [31]:
payment_first = payments.sort_values(
    ["order_id", "payment_sequential"]
).drop_duplicates(
    subset="order_id",
    keep="first"
)
chi_data = orders[
    [
        "order_id",
        "order_status"
    ]
].merge(
    payment_first[
        [
            "order_id",
            "payment_type"
        ]
    ],
    on="order_id",
    how="inner"
)
chi_data.head()

contingency_table = pd.crosstab(
    chi_data["payment_type"],
    chi_data["order_status"]
)

contingency_table

chi2_statistic, p_value, degrees_of_freedom, expected = (
    stats.chi2_contingency(contingency_table)
)
print("Chi-square statistic:", chi2_statistic)
print("Degrees of freedom:", degrees_of_freedom)
print("P-value:", p_value)
alpha = 0.05

if p_value < alpha:
    print(
        "Result: There is a statistically significant "
        "association between payment type and order status."
    )
else:
    print(
        "Result: There is no statistically significant "
        "association between payment type and order status."
    )

Chi-square statistic: 939.509784002784
Degrees of freedom: 28
P-value: 8.714660119444278e-180
Result: There is a statistically significant association between payment type and order status.


In [37]:
statistical_results = pd.DataFrame({
    "Test": [
        "Independent T-Test",
        "One-Way ANOVA",
        "Chi-Square Test"
    ],
    "Statistic": [
        t_statistic,
        f_statistic,
        chi2_statistic
    ],
    "P-Value": [
        t_test_data.shape[0] * 0 + p_value
        if False else np.nan,
        np.nan,
        np.nan
    ]
})

t_statistic, t_test_p_value = stats.ttest_ind(
    on_time_reviews,
    late_reviews,
    equal_var=False
)
f_statistic, anova_p_value = stats.f_oneway(
    *anova_groups
)
chi2_statistic, chi_square_p_value, degrees_of_freedom, expected = (
    stats.chi2_contingency(contingency_table)
)
statistical_results = pd.DataFrame({
    "Test": [
        "Independent T-Test",
        "One-Way ANOVA",
        "Chi-Square Test"
    ],
    "Statistic": [
        t_statistic,
        f_statistic,
        chi2_statistic
    ],
    "P-Value": [
        t_test_p_value,
        anova_p_value,
        chi_square_p_value
    ]
})

statistical_results
statistical_results["Significant_at_0.05"] = (
    statistical_results["P-Value"] < 0.05
)

statistical_results

,Test,Statistic,P-Value,Significant_at_0.05
0,Independent T-Test,89.550814,0.000000e+00,True
1,One-Way ANOVA,310.298758,0.000000e+00,True
2,Chi-Square Test,939.509784,8.714660e-180,True


In [39]:
import os

os.makedirs(
    "../Data/cleaned",
    exist_ok=True
)
statistical_results.to_csv(
    "../Data/cleaned/statistical_results.csv",
    index=False
)

print("Statistical results saved successfully!")

Statistical results saved successfully!


# Statistical Analysis Summary

Three statistical tests were performed for the Cart2Insights project:

### 1. Independent Samples T-Test
Used to examine whether customer review scores differ between
on-time and late deliveries.

### 2. One-Way ANOVA
Used to examine whether mean product prices differ across the
selected top product categories.

### 3. Chi-Square Test of Independence
Used to examine whether payment type and order status are
statistically associated.

A significance level of 0.05 was used for all statistical tests.
The results are based on the available cleaned dataset and
indicate statistical association/difference rather than causation.